# Sheen 2018 local-magnitude summary (raw, no station corrections)

Standalone summary of the Sheen, Kang & Rhie (2018) `M_L` recalibration applied
to the 2010–2024 PhaseNet+ Ulsan catalog. Independent of the Heo branch — no
Heo overlays except in one explicit cross-check panel.

## Sheen 2018 recipe (paper §Data & Parametric Method, pp. 2749–2751)

Calibrated on 269 events (M 2.0–5.8) recorded by 89 broadband stations at
epicentral distances 10–600 km, 2001–2016. Three components measured separately,
with separate horizontal / vertical attenuation laws:

$$
M_L^{\text{Z}} = \log_{10}(A_\mathrm{mm}) + 0.5107\,\log_{10}(R/100) + 0.001699\,(R-100) + 3,
$$
$$
M_L^{\text{N,E}} = \log_{10}(A_\mathrm{mm}) + 0.5869\,\log_{10}(R/100) + 0.001680\,(R-100) + 3.
$$

Preprocessing (matched in `ml_pipeline.per_station_ml` with `attenuation_fn=ml_sheen2018`):

1. Detrend (mean + linear).
2. Bandpass **0.5–10 Hz** — cosine-tapered `(0.3, 0.5, 10, 12)` pre_filt during
   response removal approximates Sheen's 6-pole Butterworth.
3. Remove instrument response → displacement.
4. Convolve with Wood-Anderson PAZ (sensitivity 2080, Uhrhammer & Collins 1990).
5. Peak amplitude post-P, then `M_L` via the per-component formula above.
6. SNR ≥ 3 filter (stricter than Sheen's 2×; keeps fewer events).

## Hypothesis being tested

Sheen 2018 was calibrated on the broader KMA + KIGAM + KINS network (89 stations,
distances 10–600 km) on M≥2 events. Its distance reference is R = 100 km, far
outside the close-station band that biases Heo upward. Under Sheen, the post-2017
Ulsan-area densification should make the **median event ML lower**, not higher,
because: (i) the formula is calibrated for what the broader network sees, and
(ii) more stations averaged into the event median converge toward a less
close-station-biased estimate.

**Direct test**: per-year median ML across 2010–2024. Expectation under Sheen:
pre-2017 baseline ≥ post-2017 baseline, i.e. **a downward step at the
2016-09 KG densification**, opposite to the +0.2 upward step Heo shows.

## 1. Load the Sheen 2018 augmented catalog

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seismostats as sst
from seismostats.analysis import ClassicBValueEstimator, estimate_mc_maxc, estimate_mc_ks
from seismostats.utils import bin_to_precision
warnings.filterwarnings("ignore")

CAT_SHEEN = "catalog_phasenet_plus_2010_2024_blastclean_with_ml_sheen.csv"
CAT_HEO   = "catalog_phasenet_plus_2010_2024_blastclean_with_ml_heo.csv"

df = pd.read_csv(CAT_SHEEN)
df = df.rename(columns={"lat": "latitude", "lon": "longitude"})
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["magnitude"])
print(f"Sheen 2018 catalog: {len(df):,} events with ok ML")
print(f"  ML range : {df.magnitude.min():.2f} … {df.magnitude.max():.2f}")
print(f"  ML median: {df.magnitude.median():.3f}")
print(f"  time     : {df.time.min()} … {df.time.max()}")

DELTA_M = 0.1
cat = sst.Catalog(df.copy(), delta_m=DELTA_M)
cat.bin_magnitudes(DELTA_M, inplace=True)
print(f"\nSeismoStats Catalog: {len(cat):,} events, delta_m={cat.delta_m}")

# Subregion tagging — same boxes as 03.Magnitude_summary.ipynb so subregion stats are comparable.
REGION       = [128.5, 130.0, 35.3, 36.5]
ULSAN_BOX    = [129.25, 129.55, 35.6, 35.9]
GYEONGJU_BOX = [129.15, 129.25, 35.72, 35.82]
def _tag(lon, lat):
    if GYEONGJU_BOX[0] <= lon <= GYEONGJU_BOX[1] and GYEONGJU_BOX[2] <= lat <= GYEONGJU_BOX[3]:
        return "Gyeongju"
    if ULSAN_BOX[0] <= lon <= ULSAN_BOX[1] and ULSAN_BOX[2] <= lat <= ULSAN_BOX[3]:
        return "Ulsan-Fault"
    return "other"
cat["subregion"] = [_tag(r.longitude, r.latitude) for r in cat.itertuples()]
print(cat["subregion"].value_counts().to_string())

## 2. Frequency-magnitude distribution (full catalog + subregions)

Aki-Utsu MLE _b_-value, Wiemer-Wyss MAXC Mc, Clauset/Mizrahi KS Mc. Each panel
shows histogram (left) + cumulative log-y (right). _Bin generator uses integer
arithmetic to avoid the `np.arange(step=0.1)` floating-point gotcha._

Sheen reports ML correlates with Mw at slope ≈ 1.08 (his Eqs. 7–8) — so a b-value
around 0.9-1.0 here would be physically sensible. Heo (with the close-station
inflation) gave _b_ = 1.14 ± 0.02 on this same catalog.

In [ ]:
def _safe_bins(mag_min, mag_max, delta_m=0.1):
    i_lo = int(np.floor(mag_min / delta_m)); i_hi = int(np.ceil(mag_max / delta_m))
    return np.round((np.arange(i_lo, i_hi + 2) * delta_m), 6)

def _fmd(mags, ax_h, ax_c, *, title=""):
    """Two-panel FMD: histogram (left) + cumulative log-y (right). Computes _b_ at
    BOTH MAXC Mc (red dashed) AND KS Mc (blue dotted) when KS converges. Returns
    a dict with both Mc/b pairs so the summary table can quote them side by side."""
    mags = bin_to_precision(np.sort(np.asarray(mags, float)), DELTA_M)
    mc_maxc, _ = estimate_mc_maxc(mags, fmd_bin=DELTA_M)
    try:    mc_ks, _ = estimate_mc_ks(mags, delta_m=DELTA_M, p_value_pass=0.1)
    except Exception: mc_ks = None
    # b at MAXC Mc
    be_maxc = ClassicBValueEstimator()
    be_maxc.calculate(mags[mags >= mc_maxc], mc=mc_maxc, delta_m=DELTA_M)
    b_maxc, b_maxc_se = be_maxc.b_value, be_maxc.std
    # b at KS Mc (only when KS gave a value and ≥ 30 events above it)
    b_ks = b_ks_se = None
    if mc_ks is not None and (mags >= mc_ks).sum() >= 30:
        try:
            be_ks = ClassicBValueEstimator()
            be_ks.calculate(mags[mags >= mc_ks], mc=mc_ks, delta_m=DELTA_M)
            b_ks, b_ks_se = be_ks.b_value, be_ks.std
        except Exception:
            pass

    bins = _safe_bins(mags.min(), mags.max(), DELTA_M)
    # ---- histogram (left) ----
    ax_h.hist(mags, bins=bins, color="#2ca02c", alpha=0.75, edgecolor="k", linewidth=0.4)
    ax_h.axvline(mc_maxc, color="red", lw=1.2, ls="--", label=f"MAXC Mc = {mc_maxc:.2f}")
    if mc_ks is not None:
        ax_h.axvline(mc_ks, color="#1f77ff", lw=1.0, ls=":", label=f"KS Mc = {mc_ks:.2f}")
    ax_h.set(xlabel="ML (Sheen 2018)", ylabel="Event count per 0.1 bin", title=title)
    ax_h.legend(fontsize=8); ax_h.grid(alpha=0.3)

    # ---- cumulative log-y (right) — overlay BOTH MAXC and KS fits ----
    centers = bins[:-1]; n_cum = np.array([(mags >= m).sum() for m in centers])
    nz = n_cum > 0
    ax_c.scatter(centers[nz], n_cum[nz], s=15, color="black", zorder=3)
    a_maxc = np.log10((mags >= mc_maxc).sum()) + b_maxc * mc_maxc
    xline_maxc = np.linspace(mc_maxc, mags.max(), 30)
    ax_c.plot(xline_maxc, 10**(a_maxc - b_maxc * xline_maxc), color="red", lw=1.2, ls="--",
              label=f"b (MAXC) = {b_maxc:.2f} ± {b_maxc_se:.2f}")
    if b_ks is not None:
        a_ks = np.log10((mags >= mc_ks).sum()) + b_ks * mc_ks
        xline_ks = np.linspace(mc_ks, mags.max(), 30)
        ax_c.plot(xline_ks, 10**(a_ks - b_ks * xline_ks), color="#1f77ff", lw=1.0, ls=":",
                  label=f"b (KS)   = {b_ks:.2f} ± {b_ks_se:.2f}")
    ax_c.axvline(mc_maxc, color="red", lw=0.6, ls=":", alpha=0.5)
    if mc_ks is not None:
        ax_c.axvline(mc_ks, color="#1f77ff", lw=0.6, ls=":", alpha=0.5)
    ax_c.set(xlabel="ML", ylabel="N(M ≥ ML)", yscale="log", title=title)
    ax_c.legend(fontsize=8); ax_c.grid(alpha=0.3, which="both")
    return dict(mc_maxc=mc_maxc, mc_ks=mc_ks,
                b=b_maxc, b_se=b_maxc_se,            # for backwards compat (b at MAXC)
                b_maxc=b_maxc, b_maxc_se=b_maxc_se,
                b_ks=b_ks, b_ks_se=b_ks_se)

fig, axes = plt.subplots(3, 2, figsize=(11, 11), dpi=120)
r_full = _fmd(cat.magnitude, axes[0, 0], axes[0, 1],
               title=f"Full catalog — Sheen 2018 (n={len(cat):,})")
r_uf   = _fmd(cat[cat.subregion == "Ulsan-Fault"].magnitude, axes[1, 0], axes[1, 1],
               title=f"Ulsan-Fault subregion (n={(cat.subregion=='Ulsan-Fault').sum():,})")
r_gj   = _fmd(cat[cat.subregion == "Gyeongju"].magnitude, axes[2, 0], axes[2, 1],
               title=f"2016 Gyeongju box (n={(cat.subregion=='Gyeongju').sum():,})")
plt.tight_layout(); plt.show()

# Tabular summary — both Mc estimators and their b-values
print("\nFMD summary (Sheen 2018):")
print(f"  {'Population':<12}  {'Mc_MAXC':>8}  {'b(MAXC) ± SE':>16}  {'Mc_KS':>6}  {'b(KS) ± SE':>15}")
for lab, r in (("Full",        r_full),
               ("Ulsan-Fault", r_uf),
               ("Gyeongju",    r_gj)):
    mc_ks_str = f"{r['mc_ks']:.2f}" if r['mc_ks'] is not None else "N/A"
    b_ks_str = (f"{r['b_ks']:.2f} ± {r['b_ks_se']:.2f}"
                if r['b_ks'] is not None else "N/A")
    print(f"  {lab:<12}  {r['mc_maxc']:>8.2f}  "
          f"{r['b_maxc']:.2f} ± {r['b_maxc_se']:.2f}  "
          f"{mc_ks_str:>6}  {b_ks_str:>15}")

## 3. Magnitude vs time — does Sheen show the expected downward step at 2016-09?

This is the direct test of the hypothesis from the notebook intro. Three panels:

1. Scatter coloured by year, with the 2016-09 KG densification annotated and the
   KMA-published M5.8 / M5.4 / M4.5 Gyeongju benchmarks marked.
2. Per-year median ML, with a vertical line at 2016.7 (when KG online).
3. Cumulative event count per subregion vs time (sanity check that the catalog
   covers the same epochs as the Heo branch).

Read panel 2 first: if the line drops at 2016.7, Sheen behaves as expected and
no station corrections are needed.

In [ ]:
# Three independent figures (separate plt.subplots calls so each gets a full-width
# canvas; previously bundled side-by-side they were too narrow to read).

yrs = cat.time.dt.year.to_numpy()


# ---------- Sliding-window Mc (overlapping, smoothing) ----------
# Compute MAXC and KS Mc on time-sorted overlapping windows of `WIN_N` events,
# stepping by `STEP_N`. The window's centre time is the median origin time of its
# events. This shows how Mc itself evolves with the catalog density — a sharp
# rise/fall is the network-completeness footprint that the per-year-median
# alone cannot reveal.
WIN_N  = 500     # events per window — robust MAXC convergence
STEP_N = 100     # events per step — 80% overlap → smooth curves

def _sliding_mc(mags_sorted, times_sorted, win=WIN_N, step=STEP_N, delta_m=DELTA_M):
    n = len(mags_sorted)
    if n < win:
        return [], np.array([]), np.array([])
    centres, mc_maxc_arr, mc_ks_arr = [], [], []
    for i in range(0, n - win + 1, step):
        m_chunk = bin_to_precision(np.sort(np.asarray(mags_sorted[i:i + win], float)),
                                    delta_m)
        t_chunk = np.asarray(times_sorted[i:i + win])
        mc_m, _ = estimate_mc_maxc(m_chunk, fmd_bin=delta_m)
        try:
            mc_k, _ = estimate_mc_ks(m_chunk, delta_m=delta_m, p_value_pass=0.1)
        except Exception:
            mc_k = np.nan
        centres.append(pd.Timestamp(np.median(t_chunk.astype("int64")), unit="ns"))
        mc_maxc_arr.append(float(mc_m))
        mc_ks_arr.append(float(mc_k) if mc_k is not None and np.isfinite(mc_k) else np.nan)
    return centres, np.array(mc_maxc_arr), np.array(mc_ks_arr)

_ts_sorted = cat.sort_values("time")
_mc_centres, _mc_maxc, _mc_ks = _sliding_mc(_ts_sorted.magnitude.to_numpy(),
                                            _ts_sorted.time.to_numpy())
print(f"Sliding-Mc: {len(_mc_centres)} windows of {WIN_N} events, step {STEP_N}")
print(f"  Mc_MAXC over time: median={np.median(_mc_maxc):.2f}, "
      f"range {np.nanmin(_mc_maxc):.2f}…{np.nanmax(_mc_maxc):.2f}")
if np.isfinite(_mc_ks).any():
    print(f"  Mc_KS   over time: median={np.nanmedian(_mc_ks):.2f}, "
          f"range {np.nanmin(_mc_ks):.2f}…{np.nanmax(_mc_ks):.2f}")


# ---------- (a) Magnitude vs time scatter — with sliding-window Mc overlaid ----------
fig, ax_s = plt.subplots(figsize=(12, 5.0), dpi=120)
sc = ax_s.scatter(cat.time, cat.magnitude, c=yrs, cmap="viridis",
                   s=10 + np.clip(cat.magnitude, 0, 6) ** 2 * 4,
                   alpha=0.55, edgecolor="none")
ax_s.axvline(pd.to_datetime("2016-09-01"), color="0.4", lw=0.8, ls="--")
ax_s.annotate("KG densification\n(2016-09)",
               xy=(pd.to_datetime("2016-09-01"), 5.0),
               xytext=(pd.to_datetime("2013-01-01"), 5.2),
               fontsize=9, color="0.4",
               arrowprops=dict(arrowstyle="->", lw=0.7, color="0.4"))
for t, m, lab in [("2016-09-12 11:32:54", 5.8, "M5.8 (KMA)"),
                   ("2016-09-12 10:44:32", 5.4, "M5.4 (KMA)"),
                   ("2016-09-19 11:33:58", 4.5, "M4.5 (KMA)")]:
    ax_s.annotate(lab, xy=(pd.to_datetime(t), m), xytext=(20, 8),
                   textcoords="offset points", fontsize=9,
                   arrowprops=dict(arrowstyle="->", lw=0.7, color="0.4"))
ax_s.plot(_mc_centres, _mc_maxc, color="red", lw=1.6, ls="--",
           label=f"sliding Mc (MAXC, win={WIN_N})", zorder=4)
if np.isfinite(_mc_ks).any():
    ax_s.plot(_mc_centres, _mc_ks, color="white", lw=2.5, alpha=0.55, zorder=4)
    ax_s.plot(_mc_centres, _mc_ks, color="#1f77ff", lw=1.6, ls=":",
               label=f"sliding Mc (KS, win={WIN_N})", zorder=5)
ax_s.set(xlabel="Origin time (UTC)", ylabel="ML (Sheen 2018)",
          title=f"Magnitude vs time — n={len(cat):,}")
ax_s.grid(alpha=0.3); ax_s.legend(loc="upper left", fontsize=8)
fig.colorbar(sc, ax=ax_s, label="Year")
plt.tight_layout(); plt.show()


# ---------- (b) Per-year median ML + sliding-window Mc ----------
by_year_sheen = cat.groupby(yrs)["magnitude"].agg(["median", "count"])
by_year_sheen.columns = ["ml_median", "n"]
# Sliding-Mc x-coord on the calendar-year axis: fractional year of each centre time
_mc_yrs = np.array([t.year + (t.dayofyear - 1) / 365.25 for t in _mc_centres])
fig, ax_m = plt.subplots(figsize=(11, 4.8), dpi=120)
ax_m.plot(by_year_sheen.index, by_year_sheen["ml_median"], "s-", color="#2ca02c",
           lw=1.8, ms=7, label="Sheen 2018 median ML (per-year)")
ax_m.plot(_mc_yrs, _mc_maxc, color="red", lw=1.6, ls="--",
           label=f"sliding Mc (MAXC, win={WIN_N})")
if np.isfinite(_mc_ks).any():
    ax_m.plot(_mc_yrs, _mc_ks, color="#1f77ff", lw=1.6, ls=":",
               label=f"sliding Mc (KS, win={WIN_N})")
ax_m.axvline(2016.7, color="0.4", lw=0.8, ls="--")
_ymax_b = max(by_year_sheen["ml_median"].max(), np.nanmax(_mc_maxc),
              np.nanmax(_mc_ks) if np.isfinite(_mc_ks).any() else 0)
ax_m.annotate("network densification\n(KS + KG online)",
               xy=(2016.7, _ymax_b + 0.05),
               xytext=(2013.5, _ymax_b + 0.20),
               fontsize=9, color="0.4",
               arrowprops=dict(arrowstyle="->", lw=0.7, color="0.4"))
ax_m.set(xlabel="Year", ylabel="ML",
         title="Per-year median ML + sliding Mc — Sheen 2018")
ax_m.grid(alpha=0.3); ax_m.legend(fontsize=9, loc="best")
plt.tight_layout(); plt.show()


# ---------- (c) Cumulative count per subregion ----------
fig, ax_c = plt.subplots(figsize=(11, 4.5), dpi=120)
for sub, col in (("Ulsan-Fault", "#1f77b4"), ("Gyeongju", "#d62728"), ("other", "0.5")):
    s = cat[cat.subregion == sub].sort_values("time")
    ax_c.plot(s.time, np.arange(1, len(s) + 1), label=f"{sub} (n={len(s):,})",
               color=col, lw=1.4)
ax_c.set(xlabel="Origin time", ylabel="Cumulative event count",
          title="Cumulative seismicity by subregion")
ax_c.grid(alpha=0.3); ax_c.legend(fontsize=9)
plt.tight_layout(); plt.show()


# ---------- Quantitative step at 2016-09 ----------
pre  = by_year_sheen.loc[by_year_sheen.index <= 2016, "ml_median"].median()
post = by_year_sheen.loc[by_year_sheen.index >= 2017, "ml_median"].median()
print(f"\nMedian-of-yearly-medians, Sheen 2018:")
print(f"  pre-2017  (2010–2016): {pre:.3f}")
print(f"  post-2017 (2017–2024): {post:.3f}")
print(f"  step  (post − pre)   : {post - pre:+.3f}")
if post < pre - 0.05:
    print("  → DOWNWARD step (post-2017 lower) — consistent with the hypothesis")
elif post > pre + 0.05:
    print("  → UPWARD step (post-2017 higher) — same artefact as Heo")
else:
    print("  → no significant step (|Δ| ≤ 0.05) — Sheen is time-stable")

# Sliding-Mc step at the 2016-09 boundary (median of windows on each side)
_pre_mask  = np.array([t < pd.Timestamp("2016-09-01") for t in _mc_centres])
_post_mask = np.array([t >= pd.Timestamp("2017-01-01") for t in _mc_centres])
if _pre_mask.any() and _post_mask.any():
    mc_maxc_pre  = np.median(_mc_maxc[_pre_mask])
    mc_maxc_post = np.median(_mc_maxc[_post_mask])
    print(f"\nSliding Mc step around 2016-09 (median of windows on each side):")
    print(f"  Mc_MAXC  pre={mc_maxc_pre:.2f}  post={mc_maxc_post:.2f}  "
          f"step {mc_maxc_post-mc_maxc_pre:+.2f}")
    if np.isfinite(_mc_ks).any():
        mc_ks_pre  = np.nanmedian(_mc_ks[_pre_mask])
        mc_ks_post = np.nanmedian(_mc_ks[_post_mask])
        print(f"  Mc_KS    pre={mc_ks_pre:.2f}  post={mc_ks_post:.2f}  "
              f"step {mc_ks_post-mc_ks_pre:+.2f}")

### 3b. Same plots, **Ulsan-Fault subregion only**

The full-catalog view in §3 is dominated by the 2016 Gyeongju aftershock sequence
(6,460 of 15,762 events) and the 2017 Pohang sequence. For the Ulsan-Fault zone
proper (~2,900 events in the box `129.25 ≤ lon ≤ 129.55, 35.60 ≤ lat ≤ 35.90`),
the temporal pattern is sparser and the sliding window can be tighter
(`WIN_N=250`, `STEP_N=50`) for better time resolution. We render the same two
panels (magnitude-vs-time scatter + per-year median ML + sliding Mc) restricted
to the Ulsan-Fault subset.

In [ ]:
# Ulsan-Fault subregion: magnitude vs time + sliding-window Mc
WIN_N_UF  = 250        # smaller window — UF has only ~2,900 events
STEP_N_UF = 50

uf = cat[cat.subregion == "Ulsan-Fault"].sort_values("time")
print(f"Ulsan-Fault subregion: {len(uf):,} events  "
      f"({uf.time.min()} … {uf.time.max()})")

_uf_centres, _uf_mc_maxc, _uf_mc_ks = _sliding_mc(
    uf.magnitude.to_numpy(), uf.time.to_numpy(),
    win=WIN_N_UF, step=STEP_N_UF)
print(f"Sliding-Mc (UF): {len(_uf_centres)} windows of {WIN_N_UF} events, step {STEP_N_UF}")
print(f"  Mc_MAXC over time: median={np.median(_uf_mc_maxc):.2f}, "
      f"range {np.nanmin(_uf_mc_maxc):.2f}…{np.nanmax(_uf_mc_maxc):.2f}")
if np.isfinite(_uf_mc_ks).any():
    print(f"  Mc_KS   over time: median={np.nanmedian(_uf_mc_ks):.2f}, "
          f"range {np.nanmin(_uf_mc_ks):.2f}…{np.nanmax(_uf_mc_ks):.2f}")


# ---------- (a) UF scatter + sliding Mc ----------
fig, ax_s = plt.subplots(figsize=(12, 5.0), dpi=120)
yrs_uf = uf.time.dt.year.to_numpy()
sc = ax_s.scatter(uf.time, uf.magnitude, c=yrs_uf, cmap="viridis",
                   s=12 + np.clip(uf.magnitude, 0, 6) ** 2 * 6,
                   alpha=0.65, edgecolor="none")
ax_s.axvline(pd.to_datetime("2016-09-01"), color="0.4", lw=0.8, ls="--")
ax_s.annotate("KG densification\n(2016-09)",
               xy=(pd.to_datetime("2016-09-01"), uf.magnitude.max() - 0.2),
               xytext=(pd.to_datetime("2013-01-01"), uf.magnitude.max()),
               fontsize=9, color="0.4",
               arrowprops=dict(arrowstyle="->", lw=0.7, color="0.4"))
ax_s.plot(_uf_centres, _uf_mc_maxc, color="red", lw=1.6, ls="--",
           label=f"sliding Mc (MAXC, win={WIN_N_UF})", zorder=4)
if np.isfinite(_uf_mc_ks).any():
    ax_s.plot(_uf_centres, _uf_mc_ks, color="white", lw=2.5, alpha=0.55, zorder=4)
    ax_s.plot(_uf_centres, _uf_mc_ks, color="#1f77ff", lw=1.6, ls=":",
               label=f"sliding Mc (KS, win={WIN_N_UF})", zorder=5)
ax_s.set(xlabel="Origin time (UTC)", ylabel="ML (Sheen 2018)",
          title=f"Magnitude vs time — Ulsan-Fault subregion (n={len(uf):,})")
ax_s.grid(alpha=0.3); ax_s.legend(loc="upper left", fontsize=8)
fig.colorbar(sc, ax=ax_s, label="Year")
plt.tight_layout(); plt.show()


# ---------- (b) UF per-year median + sliding Mc ----------
by_year_uf = uf.groupby(yrs_uf)["magnitude"].agg(["median", "count"])
by_year_uf.columns = ["ml_median", "n"]
_uf_mc_yrs = np.array([t.year + (t.dayofyear - 1) / 365.25 for t in _uf_centres])

fig, ax_m = plt.subplots(figsize=(11, 4.8), dpi=120)
ax_m.plot(by_year_uf.index, by_year_uf["ml_median"], "s-", color="#1f77b4",
           lw=1.8, ms=7, label="Ulsan-Fault median ML (per-year)")
ax_m.plot(_uf_mc_yrs, _uf_mc_maxc, color="red", lw=1.6, ls="--",
           label=f"sliding Mc (MAXC, win={WIN_N_UF})")
if np.isfinite(_uf_mc_ks).any():
    ax_m.plot(_uf_mc_yrs, _uf_mc_ks, color="#1f77ff", lw=1.6, ls=":",
               label=f"sliding Mc (KS, win={WIN_N_UF})")
ax_m.axvline(2016.7, color="0.4", lw=0.8, ls="--")
_ymax_uf = max(by_year_uf["ml_median"].max(), np.nanmax(_uf_mc_maxc),
               np.nanmax(_uf_mc_ks) if np.isfinite(_uf_mc_ks).any() else 0)
ax_m.annotate("network densification\n(KS + KG online)",
               xy=(2016.7, _ymax_uf + 0.05),
               xytext=(2013.5, _ymax_uf + 0.25),
               fontsize=9, color="0.4",
               arrowprops=dict(arrowstyle="->", lw=0.7, color="0.4"))
ax_m.set(xlabel="Year", ylabel="ML",
         title="Per-year median ML + sliding Mc — Ulsan-Fault subregion")
ax_m.grid(alpha=0.3); ax_m.legend(fontsize=9, loc="best")
plt.tight_layout(); plt.show()


# ---------- Quantitative pre/post-2016-09 step (UF subset) ----------
uf_pre  = by_year_uf.loc[by_year_uf.index <= 2016, "ml_median"].median()
uf_post = by_year_uf.loc[by_year_uf.index >= 2017, "ml_median"].median()
print(f"\nUlsan-Fault median-of-yearly-medians:")
print(f"  pre-2017  : {uf_pre:.3f}")
print(f"  post-2017 : {uf_post:.3f}")
print(f"  step (post − pre) : {uf_post - uf_pre:+.3f}")

_pre  = np.array([t < pd.Timestamp("2016-09-01") for t in _uf_centres])
_post = np.array([t >= pd.Timestamp("2017-01-01") for t in _uf_centres])
if _pre.any() and _post.any():
    print(f"\nSliding Mc step around 2016-09 (UF, median of windows on each side):")
    print(f"  Mc_MAXC  pre={np.median(_uf_mc_maxc[_pre]):.2f}  "
          f"post={np.median(_uf_mc_maxc[_post]):.2f}  "
          f"step {np.median(_uf_mc_maxc[_post])-np.median(_uf_mc_maxc[_pre]):+.2f}")
    if np.isfinite(_uf_mc_ks).any():
        print(f"  Mc_KS    pre={np.nanmedian(_uf_mc_ks[_pre]):.2f}  "
              f"post={np.nanmedian(_uf_mc_ks[_post]):.2f}  "
              f"step {np.nanmedian(_uf_mc_ks[_post])-np.nanmedian(_uf_mc_ks[_pre]):+.2f}")

## 4. Gyeongju + Pohang benchmark events vs KMA

The published KMA magnitudes for the 2016–2018 large-event tail are the
external reference (independent of any catalog we built). M5.8 (Gyeongju
mainshock), M5.4 (Gyeongju foreshock), M4.5 (Gyeongju aftershock), M5.4 (Pohang
mainshock), M4.3 (Pohang aftershock). The Sheen recipe should recover these
closer than the close-station-biased Heo formula does.

In [ ]:
BENCH = [
    ("M5.8 Gyeongju mainshock (KMA)",   "2016-09-12 11:32:54", 5.8),
    ("M5.4 Gyeongju foreshock (KMA)",   "2016-09-12 10:44:32", 5.1),
    ("M4.5 Gyeongju aftershock (KMA)",  "2016-09-19 11:33:58", 4.5),
    ("M5.4 Pohang mainshock (KMA)",     "2017-11-15 05:29:31", 5.4),
    ("M4.6 Pohang aftershock (KMA)",    "2018-02-10 20:03:03", 4.6),
]
rows = []
for lab, t, kma_m in BENCH:
    t0 = pd.to_datetime(t)
    sub = cat[(cat.time >= t0 - pd.Timedelta("15s")) & (cat.time <= t0 + pd.Timedelta("60s"))]
    if len(sub) and sub.magnitude.notna().any():
        r = sub.loc[sub.magnitude.idxmax()]
        rows.append({"event": lab, "time_match": str(r.time)[:19], "KMA_ML": kma_m,
                      "Sheen_ML": float(r.magnitude),
                      "Δ (Sheen − KMA)": float(r.magnitude) - kma_m})
    else:
        rows.append({"event": lab, "time_match": "NO MATCH", "KMA_ML": kma_m,
                      "Sheen_ML": np.nan, "Δ (Sheen − KMA)": np.nan})
bench_df = pd.DataFrame(rows)
print(bench_df.round(2).to_string(index=False))
matched = bench_df["Δ (Sheen − KMA)"].dropna()
if len(matched):
    print(f"\nmean   Δ (Sheen − KMA) over {len(matched)} matched events: {matched.mean():+.3f}")
    print(f"median Δ                                  :                 {matched.median():+.3f}")
    print(f"MAD (|Δ|, robust)                         :                 {matched.abs().median():.3f}")

## 5. Within-year stratification by station count

**What this plot shows**: within each calendar year, events are split into three
groups by `n_used` (the number of station-channels that contributed amplitudes to
that event's ML) — low (≤ 12), mid (13–24), high (≥ 25). For each group, the
median ML is plotted against year.

**What it tests**: if the magnitude scale is intrinsic to the source, an M2
earthquake should have ML ≈ 2 whether 10 or 100 stations recorded it. If the
three lines coincide in a given year, the ML is **network-density-invariant**.
If they spread apart, ML depends on which stations contributed — a network
artefact, not a property of the seismicity.

Heo's raw catalog showed a ~0.15-ML spread between the n_used bins in most years.
If Sheen — calibrated on a broader network — shows tighter convergence between
the lines, it confirms the formula handles the densification correctly without
needing per-station corrections.

In [ ]:
def _qbin(n):
    if n <= 12:   return "low (≤12)"
    if n <= 24:   return "mid (13–24)"
    return "high (≥25)"

strat = pd.DataFrame({"year":   yrs,
                       "n_used": cat["n_used"].to_numpy(),
                       "ml":     cat["magnitude"].to_numpy()})
strat["nbin"] = strat["n_used"].apply(_qbin)
ORDER = ["low (≤12)", "mid (13–24)", "high (≥25)"]
COL   = {"low (≤12)": "#888888", "mid (13–24)": "#fdae6b", "high (≥25)": "#e31a1c"}

fig, ax = plt.subplots(figsize=(9, 4.4), dpi=120)
for nbin in ORDER:
    s = strat[strat.nbin == nbin].groupby("year")["ml"].median()
    ax.plot(s.index, s.to_numpy(), "o-", color=COL[nbin], lw=1.5, ms=6, label=nbin)
ax.axvline(2016.7, color="0.4", lw=0.8, ls="--")
ax.set(xlabel="Year", ylabel="Median ML per (year × n_used-bin)",
       title="Within-year stratification by station count — Sheen 2018")
ax.grid(alpha=0.3); ax.legend(fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()

# Quantitative across-strata spread — the smaller, the more network-invariant
g = strat.groupby(["year", "nbin"])["ml"].median().unstack("nbin")
spread = (g.max(axis=1) - g.min(axis=1))
print(f"\nacross-strata median-ML spread per year (max − min over n_used bins):")
print(spread.round(3).to_string())
print(f"\nmedian over years: {spread.median():.3f}")
print("  (Heo's raw spread was ~0.15; closer to 0 = more network-density-invariant)")

## 6. Magnitude–depth + depth-split b-value (Sheen)

In [ ]:
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(8.5, 6), dpi=120)
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
               wspace=0.05, hspace=0.05, figure=fig)
ax  = fig.add_subplot(gs[1, 0])
axx = fig.add_subplot(gs[0, 0], sharex=ax)
axy = fig.add_subplot(gs[1, 1], sharey=ax)
ax.scatter(cat.magnitude, cat.depth, s=6, c="#2ca02c", alpha=0.35, edgecolor="none")
ax.invert_yaxis()
axx.hist(cat.magnitude, bins=_safe_bins(-1.0, 6.5, DELTA_M), color="#2ca02c", alpha=0.7)
axy.hist(cat.depth.dropna(), bins=np.arange(0, 51, 1), orientation="horizontal",
          color="#2ca02c", alpha=0.7)
ax.set(xlabel="ML (Sheen 2018)", ylabel="Depth (km)")
axx.tick_params(labelbottom=False); axy.tick_params(labelleft=False)
axx.grid(alpha=0.3); axy.grid(alpha=0.3); ax.grid(alpha=0.3)
plt.suptitle(f"Magnitude–depth joint distribution — Sheen 2018 (n={len(cat):,})", y=0.93)
plt.show()

SHALLOW_KM = 10.0
shallow = cat[cat.depth < SHALLOW_KM]
deep    = cat[cat.depth >= SHALLOW_KM]
fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=120)
r_sh = _fmd(shallow.magnitude, axes[0, 0], axes[0, 1],
             title=f"Shallow (< {SHALLOW_KM:.0f} km), Sheen — n={len(shallow):,}")
r_dp = _fmd(deep.magnitude,    axes[1, 0], axes[1, 1],
             title=f"Deep (≥ {SHALLOW_KM:.0f} km), Sheen — n={len(deep):,}")
plt.tight_layout(); plt.show()
print(f"\nshallow (<10 km, Sheen): Mc={r_sh['mc_maxc']:.2f}  b={r_sh['b']:.2f} ± {r_sh['b_se']:.2f}")
print(f"deep    (≥10 km, Sheen): Mc={r_dp['mc_maxc']:.2f}  b={r_dp['b']:.2f} ± {r_dp['b_se']:.2f}")

## 7. Size-scaled seismicity map (PyGMT, Sheen ML)

Same map style as `03.Magnitude_summary.ipynb` §5, but using Sheen-magnitudes for
marker size. Background colour = depth (viridis_r). Fault traces overlaid.

In [ ]:
import pygmt as pmt
FAULT_TRACE = "/home/msseo/from_PAGO/21.230822_SRC_Workshop/map-fig2/Map2/ss.txt"

def _mag_size_cm(mag, scale=0.04, max_cm=0.45):
    mag = np.asarray(mag, dtype=float)
    raw = scale * np.exp(0.4 * np.clip(mag, -1, 6))
    return np.minimum(raw, max_cm)

def _plot_faults(fig):
    if not os.path.exists(FAULT_TRACE): return
    lines = open(FAULT_TRACE).readlines(); seg = []; segs = []
    for i in range(len(lines)):
        if lines[i].startswith(">"): seg = []
        elif i == len(lines) - 1: break
        else:
            seg.append([float(n) for n in lines[i].split()])
            if lines[i + 1].startswith(">"): segs.append(seg)
    for s in segs:
        d = pd.DataFrame(s); fig.plot(x=d[1], y=d[0], pen="1p,black")

def sized_epicenter_map(c, reg, title, dmax=50.0):
    df_ = c.copy().sort_values("magnitude", ascending=True)
    sz = _mag_size_cm(df_.magnitude.to_numpy())
    fig = pmt.Figure()
    pmt.config(FORMAT_GEO_MAP="ddd.x", MAP_FRAME_TYPE="plain")
    fig.basemap(region=reg, projection="M15c", frame=["a", f"+t{title}"])
    fig.coast(land="white", water="lightblue", shorelines=True)
    pmt.makecpt(cmap="viridis", series=[0.0, dmax], reverse=True)
    _plot_faults(fig)
    fig.plot(x=df_.longitude, y=df_.latitude, fill=df_.depth, cmap=True,
              size=sz, style="cc", pen="0.15p,black")
    for box, col in ((ULSAN_BOX, "blue"), (GYEONGJU_BOX, "red")):
        bl = [box[0], box[1], box[1], box[0], box[0]]
        ba = [box[2], box[2], box[3], box[3], box[2]]
        fig.plot(x=bl, y=ba, pen=f"1.4p,{col},solid")
    fig.colorbar(frame=["x+lDepth (km)"])
    with fig.inset(position="jBR+w5c+o0.15c", margin=0, box="+p1p,black+gwhite"):
        fig.basemap(region=[0, 5, 0, 5], projection="X5c/5c", frame=0)
        fig.text(x=2.5, y=4.75, text="ML size key", font="9p,Helvetica-Bold,black")
        for k, m in enumerate([1, 2, 3, 4, 5]):
            y = 4.0 - 0.75 * k
            fig.plot(x=[1.2], y=[y], size=[_mag_size_cm(m)], style="cc",
                      fill="lightgray", pen="0.4p,black")
            fig.text(x=2.8, y=y, text=f"ML {m}", font="9p,Helvetica,black")
    return fig

y0, y1 = int(cat.time.dt.year.min()), int(cat.time.dt.year.max())
fig_full = sized_epicenter_map(cat, REGION, f"Size-scaled seismicity {y0}-{y1} — Sheen 2018 ML")
fig_full.show(width=900)

## 8. One explicit cross-check vs Heo (raw, no corrections)

Side-by-side per-year median: Sheen (green) vs Heo raw (blue). Same events, same
preprocessing path, only the attenuation formula + bandpass differ. The shape of
the divergence tells you which formula better handles the network change.

In [ ]:
if not os.path.exists(CAT_HEO):
    print(f"[skip] {CAT_HEO} missing — run the Heo bulk-ML to enable the cross-check.")
else:
    dh = pd.read_csv(CAT_HEO)
    dh["time"] = pd.to_datetime(dh["time"], errors="coerce")
    dh = dh.dropna(subset=["magnitude"])
    heo_by_year = dh.groupby(dh.time.dt.year)["magnitude"].median()

    fig, ax = plt.subplots(figsize=(9, 4.4), dpi=120)
    ax.plot(by_year_sheen.index, by_year_sheen["ml_median"], "s-", color="#2ca02c",
             lw=1.7, ms=6, label="Sheen 2018")
    ax.plot(heo_by_year.index, heo_by_year, "o-", color="#1f77b4",
             lw=1.7, ms=6, label="Heo 2024 (raw, no corrections)")
    ax.axvline(2016.7, color="0.4", lw=0.8, ls="--")
    ax.annotate("network densification\n(KS + KG online)",
                 xy=(2016.7, max(by_year_sheen["ml_median"].max(), heo_by_year.max()) + 0.05),
                 xytext=(2014.0, max(by_year_sheen["ml_median"].max(), heo_by_year.max()) + 0.15),
                 fontsize=8, color="0.4",
                 arrowprops=dict(arrowstyle="->", lw=0.6, color="0.4"))
    ax.set(xlabel="Year", ylabel="Median ML",
            title="Per-year median — Sheen 2018 vs Heo 2024 raw")
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()

    sheen_pre  = by_year_sheen.loc[by_year_sheen.index <= 2016, "ml_median"].median()
    sheen_post = by_year_sheen.loc[by_year_sheen.index >= 2017, "ml_median"].median()
    heo_pre    = heo_by_year.loc[heo_by_year.index <= 2016].median()
    heo_post   = heo_by_year.loc[heo_by_year.index >= 2017].median()
    print(f"\n2016-09 step (post-pre median-of-yearly-medians):")
    print(f"  Sheen 2018       : {sheen_post - sheen_pre:+.3f}")
    print(f"  Heo 2024 (raw)   : {heo_post - heo_pre:+.3f}")

## Summary — Sheen 2018 raw (no station corrections)

| Population        | n     | Mc_MAXC | b ± SE        |
|---|---|---|---|
| Full catalog      | from §2 | … | … |
| Ulsan-Fault zone  | from §2 | … | … |
| 2016 Gyeongju box | from §2 | … | … |
| Shallow (< 10 km) | from §6 | … | … |
| Deep (≥ 10 km)    | from §6 | … | … |

**Headline checks** (set by running this notebook):
- 2016-09 step in median ML (§3) — direction tells you whether Sheen is
  network-density-invariant on this catalog.
- Gyeongju benchmark MAD (§4) — closer to 0 means Sheen recovers KMA's large-M
  events accurately.
- Within-year n_used stratification (§5) — convergence between the three lines
  means the formula handles network density correctly.
- Heo-vs-Sheen cross-check (§8) — direct visual comparison of the per-year median
  curves.